In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

In [ ]:
def dj_query(num_qubits):
    qc = QuantumCircuit(num_qubits + 1)
    if np.random.randint(0, 2):
        # Flip output qubit with 50% chance.
        qc.x(num_qubits)
    if np.random.randint(0, 2):
        # return constant circuit with 50% chance.
        return qc

    # Choose half the possible input strings.
    on_state = np.random.choice(range(2 ** num_qubits), 2 ** num_qubits // 2, replace=False)

    def add_cx(qc, bitstring):
        for qubit, bit in enumerate(reversed(bitstring)):
            if bit == "1":
                qc.x(qubit)
        return qc

    for state in on_state:
        qc.barrier()
        qc = add_cx(qc, f"{state:0b}")
        qc.mcx(list(range(num_qubits)), num_qubits) #controlが全て1の時、targetをflip
        qc = add_cx(qc, f"{state:0b}")

    qc.barrier()
    return qc

In [ ]:
display(dj_query(3).draw(output="mpl"))

In [ ]:
def compile_circuit(function: QuantumCircuit):
    n = function.num_qubits - 1
    qc = QuantumCircuit(n+1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.compose(function, inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

In [ ]:
def dj_algorithm(function: QuantumCircuit):
    qc = compile_circuit(function)
    result = AerSimulator().run(qc, shots=1, memory=True).result()
    mesurements = result.get_memory()
    if "1" in mesurements[0]:
        return "balanced"
    return "constant"


In [ ]:
f = dj_query(3)
display(f.draw("mpl"))
display(dj_algorithm(f))